# Notebook 03: Robust Rank Aggregation (RRA) Meta-Analysis + Convergence Filter + Sensitivity Analysis

**Input**: DE results from notebooks 01 and 02
**Output**:
- `meta_analysis_RRA_GSE50838.csv` — Full RRA scores for all genes
- `convergent_genes_RRA_GSE50838_20genes.csv` — 20 convergent SIEM genes
- Sensitivity analysis across thresholds: 0.5, 0.75, 1.0, 1.5

## RRA Method (Kolde et al., 2012)

RRA_i = exp((1/k) * sum(log(r_ij)))

where r_ij is the normalized rank of gene i in dataset j.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = '../tables'
FIGURE_DIR = '../figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print('Notebook 03: RRA Meta-Analysis + Convergence + Sensitivity')

## Step 1: Load DE Results from All Three Datasets

In [ ]:
# Load results
nk = pd.read_csv(f'{OUTPUT_DIR}/NK_IL2_GSE50838_differential_expression.csv')
nk = nk.rename(columns={'gene': 'gene_symbol'})

tnf = pd.read_csv(f'{OUTPUT_DIR}/Neut_TNF_GSE40548_differential_expression.csv')
lps = pd.read_csv(f'{OUTPUT_DIR}/Neut_LPS_GSE221866_differential_expression.csv')

# Find common genes
common = set(nk['gene_symbol']) & set(tnf['gene_symbol']) & set(lps['gene_symbol'])
print(f'Common genes across all 3 datasets: {len(common):,}')

# Filter to common genes
nk_c = nk[nk['gene_symbol'].isin(common)].set_index('gene_symbol')
tnf_c = tnf[tnf['gene_symbol'].isin(common)].set_index('gene_symbol')
lps_c = lps[lps['gene_symbol'].isin(common)].set_index('gene_symbol')

# Merge
meta = pd.DataFrame(index=sorted(common))
meta['NK_log2FC'] = nk_c['log2FC']
meta['NK_FDR'] = nk_c['FDR']
meta['TNF_log2FC'] = tnf_c['log2FC']
meta['TNF_FDR'] = tnf_c['FDR']
meta['LPS_log2FC'] = lps_c['log2FC']
meta['LPS_FDR'] = lps_c['FDR']
meta = meta.dropna()

print(f'Merged matrix: {meta.shape[0]} genes x {meta.shape[1]} columns')

## Step 2: Apply RRA — Rank Aggregation Across Datasets

In [ ]:
# Rank within each dataset (by absolute FC, descending)
meta['NK_rank'] = rankdata(-meta['NK_log2FC'].abs())
meta['TNF_rank'] = rankdata(-meta['TNF_log2FC'].abs())
meta['LPS_rank'] = rankdata(-meta['LPS_log2FC'].abs())

# RRA score = geometric mean of ranks
# Lower RRA = higher ranked (more consistently regulated)
n_genes = len(meta)
meta['RRA_score'] = np.exp(np.mean([
    np.log(meta['NK_rank'] / n_genes),
    np.log(meta['TNF_rank'] / n_genes),
    np.log(meta['LPS_rank'] / n_genes)
], axis=0))

# Benjamini-Hochberg FDR on RRA scores
from statsmodels.stats.multitest import multipletests
meta['RRA_FDR'] = multipletests(meta['RRA_score'], method='fdr_bh')[1]

# Sort by RRA score
meta = meta.sort_values('RRA_score')

# Save full RRA results
meta.to_csv(f'{OUTPUT_DIR}/meta_analysis_RRA_GSE50838.csv')
print(f'RRA scores computed for {len(meta)} genes')
print(f'\nTop 10 by RRA:')
print(meta.head(10)[['NK_log2FC', 'TNF_log2FC', 'LPS_log2FC', 'RRA_score']].round(3).to_string())

## Step 3: Convergence Filter — |log2FC| > 1, Same Direction, All 3 Datasets

In [ ]:
# Convergence criteria
# 1. |log2FC| > 1 in ALL three datasets
# 2. Same direction (all up OR all down)

convergent = meta[
    (meta['NK_log2FC'].abs() > 1) &
    (meta['TNF_log2FC'].abs() > 1) &
    (meta['LPS_log2FC'].abs() > 1) &
    (
        ((meta['NK_log2FC'] > 0) & (meta['TNF_log2FC'] > 0) & (meta['LPS_log2FC'] > 0)) |
        ((meta['NK_log2FC'] < 0) & (meta['TNF_log2FC'] < 0) & (meta['LPS_log2FC'] < 0))
    )
].copy()

# Annotate direction
convergent['direction'] = convergent['NK_log2FC'].apply(lambda x: 'UP' if x > 0 else 'DOWN')

# Add gene descriptions (basic)
gene_desc = {
    'F3': 'Tissue factor (coagulation)',
    'PLAU': 'Urokinase (fibrinolysis)',
    'BATF3': 'AP-1 transcription factor',
    'CD40': 'Costimulatory receptor',
    'CD274': 'PD-L1 (immune checkpoint)',
    'IRF5': 'Interferon regulatory factor 5',
    'SDC4': 'Syndecan-4 (adhesion)',
    'CKAP2L': 'Cytoskeleton-associated',
    'MRPL24': 'Mitochondrial ribosomal protein',
    'TFEC': 'Transcription factor EC',
    'ACAT2': 'Acetyl-CoA transferase',
    'BYSL': 'Bystin (cell adhesion)',
    'TMEM120B': 'Transmembrane protein',
    'TTL': 'Tubulin tyrosine ligase',
    'ENC1': 'Ectodermal-neural cortex 1',
    'RGS2': 'G-protein signaling regulator',
    'THBD': 'Thrombomodulin (anticoagulant)',
    'HEY1': 'Notch target transcription factor',
    'TLR1': 'Toll-like receptor 1',
    'PPP1R3B': 'Glycogen metabolism'
}
convergent['description'] = convergent.index.map(gene_desc)

# Sort: upregulated first, then by abs FC
convergent['abs_FC_mean'] = convergent[['NK_log2FC', 'TNF_log2FC', 'LPS_log2FC']].abs().mean(axis=1)
convergent = convergent.sort_values(['direction', 'abs_FC_mean'], ascending=[False, False])

# Save
convergent.to_csv(f'{OUTPUT_DIR}/convergent_genes_RRA_GSE50838_20genes.csv')

print(f'\n=== CONVERGENT SIEM GENES (n={len(convergent)}) ===')
print(f'Upregulated: {(convergent["direction"] == "UP").sum()}')
print(f'Downregulated: {(convergent["direction"] == "DOWN").sum()}')
print(f'\nGene list: {list(convergent.index)}')

# Display table
display_cols = ['NK_log2FC', 'TNF_log2FC', 'LPS_log2FC', 'NK_FDR', 'TNF_FDR', 'LPS_FDR', 'direction', 'description']
print(f'\n{convergent[display_cols].round(3).to_string()}')

## Step 4: Sensitivity Analysis — Multiple |log2FC| Thresholds

In [ ]:
thresholds = [0.5, 0.75, 1.0, 1.5]
results = []

for thresh in thresholds:
    conv = meta[
        (meta['NK_log2FC'].abs() > thresh) &
        (meta['TNF_log2FC'].abs() > thresh) &
        (meta['LPS_log2FC'].abs() > thresh) &
        (
            ((meta['NK_log2FC'] > 0) & (meta['TNF_log2FC'] > 0) & (meta['LPS_log2FC'] > 0)) |
            ((meta['NK_log2FC'] < 0) & (meta['TNF_log2FC'] < 0) & (meta['LPS_log2FC'] < 0))
        )
    ]
    n_up = ((conv['NK_log2FC'] > 0)).sum()
    n_down = ((conv['NK_log2FC'] < 0)).sum()
    results.append({
        'threshold': f'> {thresh}',
        'up': int(n_up),
        'down': int(n_down),
        'total': len(conv),
        'genes': ', '.join(list(conv.index)[:10]) + ('...' if len(conv) > 10 else '')
    })

sens_df = pd.DataFrame(results)
print('=== Sensitivity Analysis ===')
print(sens_df[['threshold', 'up', 'down', 'total']].to_string(index=False))

# Save
sens_df.to_csv(f'{OUTPUT_DIR}/sensitivity_analysis_thresholds.csv', index=False)
print(f'\nSaved: sensitivity_analysis_thresholds.csv')

## Step 5: Pairwise Correlation of Fold-Changes

In [ ]:
from scipy.stats import pearsonr

# Pearson correlations
r_nk_tnf, p_nk_tnf = pearsonr(meta['NK_log2FC'], meta['TNF_log2FC'])
r_nk_lps, p_nk_lps = pearsonr(meta['NK_log2FC'], meta['LPS_log2FC'])
r_tnf_lps, p_tnf_lps = pearsonr(meta['TNF_log2FC'], meta['LPS_log2FC'])

# Correlation on convergent genes only
conv_genes = convergent.index
r_nk_tnf_c, _ = pearsonr(meta.loc[conv_genes, 'NK_log2FC'], meta.loc[conv_genes, 'TNF_log2FC'])
r_nk_lps_c, _ = pearsonr(meta.loc[conv_genes, 'NK_log2FC'], meta.loc[conv_genes, 'LPS_log2FC'])
r_tnf_lps_c, _ = pearsonr(meta.loc[conv_genes, 'TNF_log2FC'], meta.loc[conv_genes, 'LPS_log2FC'])

print('=== Pairwise Pearson Correlations ===')
print(f'All genes (n={len(meta)}):')
print(f'  NK ↔ TNF-alpha: r = {r_nk_tnf:.3f}, p = {p_nk_tnf:.2e}')
print(f'  NK ↔ LPS:       r = {r_nk_lps:.3f}, p = {p_nk_lps:.2e}')
print(f'  TNF ↔ LPS:      r = {r_tnf_lps:.3f}, p = {p_tnf_lps:.2e}')
print(f'\nConvergent genes only (n={len(conv_genes)}):')
print(f'  NK ↔ TNF-alpha: r = {r_nk_tnf_c:.3f}')
print(f'  NK ↔ LPS:       r = {r_nk_lps_c:.3f}')
print(f'  TNF ↔ LPS:      r = {r_tnf_lps_c:.3f}')